# ClinVar Germline Variant Annotation

## Purpose

This notebook analyzes public ClinVar variant summary data to explore germline variant annotation patterns across selected clinically relevant genes.

The goal is to demonstrate a reproducible clinical genomics workflow using public data, including variant filtering, clinical significance summarization, review status assessment, and educational prioritization.

This analysis is for educational and portfolio purposes only. It does not perform clinical variant classification and should not be used for diagnosis, treatment decisions, or clinical reporting.

In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Define project paths
project_dir = Path("..")
data_dir = project_dir / "data"
results_dir = project_dir / "results"
clinvar_dir = data_dir / "clinvar"

# Create results folder if needed
results_dir.mkdir(parents=True, exist_ok=True)

clinvar_path = clinvar_dir / "variant_summary.txt.gz"

clinvar_path

WindowsPath('../data/clinvar/variant_summary.txt.gz')

In [2]:
clinvar = pd.read_csv(
    clinvar_path,
    sep="\t",
    low_memory=False
)

clinvar.shape

(8978989, 43)

In [3]:
clinvar.head()

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
0,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,...,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
1,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,...,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
2,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,G,-,-,-,-,-,-,SCV000020156,-,-
3,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,G,-,-,-,-,-,-,SCV000020156,-,-
4,15043,single nucleotide variant,NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg),9640,ZNF592,HGNC:28986,Uncertain significance,0,"Jun 29, 2015",150829393,...,A,-,-,-,-,-,-,SCV000020157,-,-


In [4]:
clinvar.columns.tolist()

['#AlleleID',
 'Type',
 'Name',
 'GeneID',
 'GeneSymbol',
 'HGNC_ID',
 'ClinicalSignificance',
 'ClinSigSimple',
 'LastEvaluated',
 'RS# (dbSNP)',
 'nsv/esv (dbVar)',
 'RCVaccession',
 'PhenotypeIDS',
 'PhenotypeList',
 'Origin',
 'OriginSimple',
 'Assembly',
 'ChromosomeAccession',
 'Chromosome',
 'Start',
 'Stop',
 'ReferenceAllele',
 'AlternateAllele',
 'Cytogenetic',
 'ReviewStatus',
 'NumberSubmitters',
 'Guidelines',
 'TestedInGTR',
 'OtherIDs',
 'SubmitterCategories',
 'VariationID',
 'PositionVCF',
 'ReferenceAlleleVCF',
 'AlternateAlleleVCF',
 'SomaticClinicalImpact',
 'SomaticClinicalImpactLastEvaluated',
 'ReviewStatusClinicalImpact',
 'Oncogenicity',
 'OncogenicityLastEvaluated',
 'ReviewStatusOncogenicity',
 'SCVsForAggregateGermlineClassification',
 'SCVsForAggregateSomaticClinicalImpact',
 'SCVsForAggregateOncogenicityClassification']

## Dataset Overview

The ClinVar variant summary file was loaded as a compressed tab-delimited file. This dataset contains public ClinVar records with genomic locations and associated variant-level metadata.

The initial dataset contains millions of records, so the next steps will focus on selected clinically relevant genes and fields useful for educational germline variant annotation.

In [5]:
key_columns = [
    "#AlleleID",
    "Type",
    "Name",
    "GeneID",
    "GeneSymbol",
    "ClinicalSignificance",
    "ClinSigSimple",
    "LastEvaluated",
    "ReviewStatus",
    "Chromosome",
    "Start",
    "Stop",
    "ReferenceAllele",
    "AlternateAllele",
    "Assembly",
    "VariationID"
]

available_key_columns = [col for col in key_columns if col in clinvar.columns]
missing_key_columns = [col for col in key_columns if col not in clinvar.columns]

print("Available key columns:")
print(available_key_columns)

print("\nMissing key columns:")
print(missing_key_columns)

Available key columns:
['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'ClinicalSignificance', 'ClinSigSimple', 'LastEvaluated', 'ReviewStatus', 'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele', 'Assembly', 'VariationID']

Missing key columns:
[]


## Select Clinically Relevant Columns

The full ClinVar summary file contains many columns. For this educational germline annotation workflow, the analysis will focus on columns related to variant identity, gene context, genomic position, clinical significance, review status, and reference assembly.

In [6]:
# Select columns most relevant to this germline annotation workflow
clinvar_columns = [
    "#AlleleID",
    "VariationID",
    "Type",
    "Name",
    "GeneID",
    "GeneSymbol",
    "HGNC_ID",
    "ClinicalSignificance",
    "ClinSigSimple",
    "LastEvaluated",
    "ReviewStatus",
    "RS# (dbSNP)",
    "PhenotypeList",
    "Origin",
    "OriginSimple",
    "Assembly",
    "Chromosome",
    "Start",
    "Stop",
    "ReferenceAllele",
    "AlternateAllele",
]

clinvar_subset = clinvar[clinvar_columns].copy()

clinvar_subset.shape

(8978989, 21)

In [7]:
clinvar_subset.head()

,#AlleleID,VariationID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,...,RS# (dbSNP),PhenotypeList,Origin,OriginSimple,Assembly,Chromosome,Start,Stop,ReferenceAllele,AlternateAllele
0,15041,2,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",...,397704705,Hereditary spastic paraplegia 48|Macular dystr...,germline;unknown,germline,GRCh37,7,4820844,4820847,na,na
1,15041,2,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",...,397704705,Hereditary spastic paraplegia 48|Macular dystr...,germline;unknown,germline,GRCh38,7,4781213,4781216,na,na
2,15042,3,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",...,397704709,Hereditary spastic paraplegia 48,germline,germline,GRCh37,7,4827361,4827374,na,na
3,15042,3,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",...,397704709,Hereditary spastic paraplegia 48,germline,germline,GRCh38,7,4787730,4787743,na,na
4,15043,4,single nucleotide variant,NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg),9640,ZNF592,HGNC:28986,Uncertain significance,0,"Jun 29, 2015",...,150829393,Galloway-Mowat syndrome 1,germline,germline,GRCh37,15,85342440,85342440,na,na


In [8]:
clinvar_subset["Assembly"].value_counts(dropna=False)

Assembly
GRCh37    4508975
GRCh38    4455872
na           9371
NCBI36       4771
Name: count, dtype: int64

In [9]:
clinvar_grch38 = clinvar_subset[clinvar_subset["Assembly"] == "GRCh38"].copy()

print(f"GRCh38 ClinVar records: {clinvar_grch38.shape[0]:,}")
print(f"Columns: {clinvar_grch38.shape[1]:,}")

GRCh38 ClinVar records: 4,455,872
Columns: 21


## Filter to a Clinically Relevant Germline Gene Panel

To keep the analysis focused and interpretable, this workflow uses a selected gene panel spanning several areas of germline clinical genetics, including hereditary breast and ovarian cancer, Lynch syndrome, colorectal cancer predisposition, familial hypercholesterolemia, cardiomyopathy, cystic fibrosis, and hereditary hearing loss.

In [10]:
gene_panel = [
    # Hereditary breast and ovarian cancer
    "BRCA1", "BRCA2",

    # Lynch syndrome / colorectal cancer predisposition
    "MLH1", "MSH2", "MSH6", "PMS2", "EPCAM", "APC", "MUTYH",

    # Familial hypercholesterolemia
    "LDLR", "APOB", "PCSK9",

    # Cardiomyopathy
    "MYH7", "MYBPC3", "TNNT2", "TNNI3",

    # Cystic fibrosis
    "CFTR",

    # Hereditary hearing loss
    "GJB2",
]

clinvar_panel = clinvar_grch38[
    clinvar_grch38["GeneSymbol"].isin(gene_panel)
].copy()

print(f"Selected gene panel records: {clinvar_panel.shape[0]:,}")
print(f"Selected genes represented: {clinvar_panel['GeneSymbol'].nunique():,}")

Selected gene panel records: 120,055
Selected genes represented: 18


In [11]:
gene_counts = (
    clinvar_panel["GeneSymbol"]
    .value_counts()
    .rename_axis("GeneSymbol")
    .reset_index(name="Variant_Record_Count")
)

gene_counts

,GeneSymbol,Variant_Record_Count
0,BRCA2,21384
1,APC,16869
2,BRCA1,15363
3,MSH6,10408
4,MSH2,8167
5,MLH1,6275
6,CFTR,6094
7,PMS2,5999
8,MYH7,5808
9,APOB,5094


## Summarize Clinical Significance

ClinVar records include clinical significance labels submitted by laboratories and expert groups. These labels are summarized here to evaluate the distribution of pathogenic, likely pathogenic, benign, uncertain, and conflicting interpretations within the selected gene panel.

In [12]:
clinical_sig_counts = (
    clinvar_panel["ClinicalSignificance"]
    .value_counts(dropna=False)
    .rename_axis("ClinicalSignificance")
    .reset_index(name="Record_Count")
)

clinical_sig_counts

,ClinicalSignificance,Record_Count
0,Uncertain significance,41924
1,Likely benign,23679
2,Pathogenic,19547
3,Conflicting classifications of pathogenicity,16682
4,Benign/Likely benign,4913
5,Likely pathogenic,3695
6,Benign,3487
7,Pathogenic/Likely pathogenic,2776
8,-,2215
9,other,884


In [13]:
clin_sig_simple_counts = (
    clinvar_panel["ClinSigSimple"]
    .value_counts(dropna=False)
    .rename_axis("ClinSigSimple")
    .reset_index(name="Record_Count")
)

clin_sig_simple_counts

,ClinSigSimple,Record_Count
0,0,89369
1,1,28455
2,-1,2231


In [14]:
gene_clinsig_counts = (
    clinvar_panel
    .groupby(["GeneSymbol", "ClinicalSignificance"])
    .size()
    .reset_index(name="Record_Count")
    .sort_values(["GeneSymbol", "Record_Count"], ascending=[True, False])
)

gene_clinsig_counts.head(30)

,GeneSymbol,ClinicalSignificance,Record_Count
9,APC,Uncertain significance,8762
7,APC,Pathogenic,2100
5,APC,Likely benign,1907
2,APC,Benign/Likely benign,1499
11,APC,other,884
3,APC,Conflicting classifications of pathogenicity,759
1,APC,Benign,414
6,APC,Likely pathogenic,262
8,APC,Pathogenic/Likely pathogenic,240
0,APC,-,39


In [15]:
pathogenic_terms = [
    "Pathogenic",
    "Likely pathogenic",
    "Pathogenic/Likely pathogenic"
]

clinvar_pathogenic = clinvar_panel[
    clinvar_panel["ClinicalSignificance"].isin(pathogenic_terms)
].copy()

print(f"Pathogenic/likely pathogenic records: {clinvar_pathogenic.shape[0]:,}")
print(f"Genes represented: {clinvar_pathogenic['GeneSymbol'].nunique():,}")

Pathogenic/likely pathogenic records: 26,018
Genes represented: 18


In [16]:
pathogenic_by_gene = (
    clinvar_pathogenic["GeneSymbol"]
    .value_counts()
    .rename_axis("GeneSymbol")
    .reset_index(name="Pathogenic_Likely_Pathogenic_Record_Count")
)

pathogenic_by_gene

,GeneSymbol,Pathogenic_Likely_Pathogenic_Record_Count
0,BRCA2,5661
1,BRCA1,4256
2,APC,2602
3,MSH6,2260
4,MSH2,2176
5,LDLR,2011
6,MLH1,1938
7,CFTR,1465
8,MYBPC3,1060
9,PMS2,991


In [17]:
gene_pathogenic_summary = gene_counts.merge(
    pathogenic_by_gene,
    on="GeneSymbol",
    how="left"
)

gene_pathogenic_summary["Pathogenic_Likely_Pathogenic_Record_Count"] = (
    gene_pathogenic_summary["Pathogenic_Likely_Pathogenic_Record_Count"]
    .fillna(0)
    .astype(int)
)

gene_pathogenic_summary["Pathogenic_Likely_Pathogenic_Percent"] = (
    gene_pathogenic_summary["Pathogenic_Likely_Pathogenic_Record_Count"]
    / gene_pathogenic_summary["Variant_Record_Count"]
    * 100
)

gene_pathogenic_summary = gene_pathogenic_summary.sort_values(
    "Pathogenic_Likely_Pathogenic_Percent",
    ascending=False
)

gene_pathogenic_summary

,GeneSymbol,Variant_Record_Count,Pathogenic_Likely_Pathogenic_Record_Count,Pathogenic_Likely_Pathogenic_Percent
11,LDLR,4558,2011,44.120228
17,GJB2,656,253,38.567073
5,MLH1,6275,1938,30.884462
2,BRCA1,15363,4256,27.702923
4,MSH2,8167,2176,26.643810
0,BRCA2,21384,5661,26.473064
6,CFTR,6094,1465,24.040039
10,MYBPC3,4658,1060,22.756548
3,MSH6,10408,2260,21.714066
7,PMS2,5999,991,16.519420


In [18]:
gene_pathogenic_summary.to_csv(
    results_dir / "clinvar_gene_pathogenic_summary.csv",
    index=False
)

## Pathogenic and Likely Pathogenic Records by Gene

Pathogenic and likely pathogenic ClinVar records were summarized across the selected germline gene panel. Raw counts were highest in heavily represented clinical testing genes, including BRCA2, BRCA1, APC, MSH6, MSH2, LDLR, MLH1, and CFTR.

Because ClinVar record counts vary substantially by gene, the proportion of pathogenic/likely pathogenic records was also calculated relative to the total number of records for each gene. This provides a normalized view of pathogenic/likely pathogenic representation within the selected panel.

These summaries reflect ClinVar-submitted clinical significance labels and are intended for educational annotation and prioritization, not independent clinical variant classification.

## Review Status and Evidence Confidence

ClinVar clinical significance labels are accompanied by review status information that reflects the level of review and evidence supporting each interpretation. This section summarizes review status across the selected gene panel and within pathogenic/likely pathogenic records.

This distinction is important because ClinVar records with expert panel review or practice guideline support generally carry more interpretive confidence than records with limited review, single-submitter evidence, or conflicting interpretations.

In [19]:
review_status_counts = (
    clinvar_panel["ReviewStatus"]
    .value_counts(dropna=False)
    .rename_axis("ReviewStatus")
    .reset_index(name="Record_Count")
)

review_status_counts

,ReviewStatus,Record_Count
0,"criteria provided, single submitter",51468
1,"criteria provided, multiple submitters, no con...",36144
2,"criteria provided, conflicting classifications",16675
3,reviewed by expert panel,10411
4,no assertion criteria provided,2900
5,-,2215
6,no classification provided,199
7,practice guideline,23
8,no classification for the single variant,16
9,no classifications from unflagged records,4


In [20]:
pathogenic_review_status_counts = (
    clinvar_pathogenic["ReviewStatus"]
    .value_counts(dropna=False)
    .rename_axis("ReviewStatus")
    .reset_index(name="Pathogenic_Likely_Pathogenic_Record_Count")
)

pathogenic_review_status_counts

,ReviewStatus,Pathogenic_Likely_Pathogenic_Record_Count
0,"criteria provided, single submitter",10733
1,"criteria provided, multiple submitters, no con...",7423
2,reviewed by expert panel,6935
3,no assertion criteria provided,904
4,practice guideline,23


In [21]:
pathogenic_review_by_gene = (
    clinvar_pathogenic
    .groupby(["GeneSymbol", "ReviewStatus"])
    .size()
    .reset_index(name="Record_Count")
    .sort_values(["GeneSymbol", "Record_Count"], ascending=[True, False])
)

pathogenic_review_by_gene.head(50)

,GeneSymbol,ReviewStatus,Record_Count
1,APC,"criteria provided, single submitter",1433
0,APC,"criteria provided, multiple submitters, no con...",1039
2,APC,no assertion criteria provided,78
3,APC,reviewed by expert panel,52
5,APOB,"criteria provided, single submitter",183
4,APOB,"criteria provided, multiple submitters, no con...",47
6,APOB,no assertion criteria provided,20
10,BRCA1,reviewed by expert panel,2271
8,BRCA1,"criteria provided, single submitter",1218
7,BRCA1,"criteria provided, multiple submitters, no con...",591


In [22]:
review_status_counts.to_csv(
    results_dir / "clinvar_review_status_counts.csv",
    index=False
)

pathogenic_review_status_counts.to_csv(
    results_dir / "clinvar_pathogenic_review_status_counts.csv",
    index=False
)

pathogenic_review_by_gene.to_csv(
    results_dir / "clinvar_pathogenic_review_status_by_gene.csv",
    index=False
)

## Educational ClinVar-Based Variant Prioritization

This section creates an educational prioritization table using ClinVar clinical significance, review status, and gene context.

This is not a clinical variant classification workflow. The prioritization score is intended only to demonstrate how public ClinVar annotations can be organized into a ranked table for exploratory review.

True clinical classification would require full ACMG/AMP evidence assessment, disease-specific gene validity, inheritance context, phenotype correlation, population frequency thresholds, segregation data, functional evidence, and expert review.

In [23]:
# Create a copy for prioritization
priority_df = clinvar_panel.copy()

# Keep a focused set of columns for the final prioritization table
priority_columns = [
    "#AlleleID",
    "VariationID",
    "GeneSymbol",
    "Name",
    "Type",
    "ClinicalSignificance",
    "ClinSigSimple",
    "ReviewStatus",
    "PhenotypeList",
    "Origin",
    "OriginSimple",
    "Assembly",
    "Chromosome",
    "Start",
    "Stop",
    "ReferenceAllele",
    "AlternateAllele",
    "RS# (dbSNP)",
    "LastEvaluated",
]

priority_columns = [col for col in priority_columns if col in priority_df.columns]
priority_df = priority_df[priority_columns].copy()

priority_df.head()

,#AlleleID,VariationID,GeneSymbol,Name,Type,ClinicalSignificance,ClinSigSimple,ReviewStatus,PhenotypeList,Origin,OriginSimple,Assembly,Chromosome,Start,Stop,ReferenceAllele,AlternateAllele,RS# (dbSNP),LastEvaluated
1501,15832,793,APC,NM_000038.5(APC):c.730_731delAG,Microsatellite,Pathogenic,1,"criteria provided, multiple submitters, no con...",Familial adenomatous polyposis 1|Hereditary ca...,germline;unknown,germline,GRCh38,5,112801277,112801278,na,na,387906228,"Dec 03, 2025"
1503,15834,795,APC,NM_000038.6(APC):c.1369del (p.Ser457fs),Deletion,Pathogenic,1,"criteria provided, multiple submitters, no con...",Familial adenomatous polyposis 1|Hereditary ca...,germline;unknown,germline,GRCh38,5,112821950,112821950,na,na,387906229,"Feb 06, 2025"
1505,15835,796,APC,NM_000038.6(APC):c.1500T>G (p.Tyr500Ter),single nucleotide variant,Pathogenic,1,"criteria provided, multiple submitters, no con...",Familial adenomatous polyposis 1|not provided|...,germline;unknown,germline,GRCh38,5,112827199,112827199,na,na,387906230,"Nov 08, 2024"
1507,15836,797,APC,NM_000038.6(APC):c.1240C>T (p.Arg414Cys),single nucleotide variant,Benign,1,reviewed by expert panel,Gardner syndrome|not provided|Hereditary cance...,germline;unknown,germline,GRCh38,5,112819272,112819272,na,na,137854567,"Feb 18, 2023"
1509,15837,798,APC,NM_000038.6(APC):c.904C>T (p.Arg302Ter),single nucleotide variant,Pathogenic/Likely pathogenic,1,"criteria provided, multiple submitters, no con...",Familial adenomatous polyposis 1|Gardner syndr...,germline;somatic;unknown,germline/somatic,GRCh38,5,112815564,112815564,na,na,137854568,"Apr 08, 2025"


In [24]:
def assign_clinsig_score(clinical_significance):
    """
    Assign an educational score based on ClinVar clinical significance.

    This is not an ACMG/AMP classification.
    It is only a prioritization score for exploratory review.
    """
    if pd.isna(clinical_significance):
        return 0

    sig = str(clinical_significance).lower()

    if "pathogenic/likely pathogenic" in sig:
        return 5
    elif sig == "pathogenic":
        return 5
    elif sig == "likely pathogenic":
        return 4
    elif "conflicting" in sig:
        return 3
    elif "uncertain significance" in sig:
        return 2
    elif sig == "likely benign":
        return 1
    elif sig == "benign":
        return 0
    else:
        return 0


def assign_review_score(review_status):
    """
    Assign an educational score based on ClinVar review status.

    Higher scores indicate stronger review status.
    """
    if pd.isna(review_status):
        return 0

    status = str(review_status).lower()

    if "practice guideline" in status:
        return 5
    elif "reviewed by expert panel" in status:
        return 4
    elif "criteria provided, multiple submitters, no conflicts" in status:
        return 3
    elif "criteria provided, conflicting classifications" in status:
        return 2
    elif "criteria provided, single submitter" in status:
        return 1
    else:
        return 0


def assign_gene_context_score(gene):
    """
    Assign a small educational score based on whether the gene belongs to
    a well-established clinical testing category in this project.
    """
    high_context_genes = {
        "BRCA1", "BRCA2",
        "MLH1", "MSH2", "MSH6", "PMS2", "EPCAM",
        "APC", "MUTYH",
        "LDLR", "APOB", "PCSK9",
        "MYH7", "MYBPC3", "TNNT2", "TNNI3",
        "CFTR", "GJB2",
    }

    if gene in high_context_genes:
        return 1
    else:
        return 0

In [25]:
priority_df["ClinicalSignificance_Score"] = priority_df["ClinicalSignificance"].apply(
    assign_clinsig_score
)

priority_df["ReviewStatus_Score"] = priority_df["ReviewStatus"].apply(
    assign_review_score
)

priority_df["GeneContext_Score"] = priority_df["GeneSymbol"].apply(
    assign_gene_context_score
)

priority_df["Educational_Priority_Score"] = (
    priority_df["ClinicalSignificance_Score"]
    + priority_df["ReviewStatus_Score"]
    + priority_df["GeneContext_Score"]
)

priority_df.head()

,#AlleleID,VariationID,GeneSymbol,Name,Type,ClinicalSignificance,ClinSigSimple,ReviewStatus,PhenotypeList,Origin,...,Start,Stop,ReferenceAllele,AlternateAllele,RS# (dbSNP),LastEvaluated,ClinicalSignificance_Score,ReviewStatus_Score,GeneContext_Score,Educational_Priority_Score
1501,15832,793,APC,NM_000038.5(APC):c.730_731delAG,Microsatellite,Pathogenic,1,"criteria provided, multiple submitters, no con...",Familial adenomatous polyposis 1|Hereditary ca...,germline;unknown,...,112801277,112801278,na,na,387906228,"Dec 03, 2025",5,3,1,9
1503,15834,795,APC,NM_000038.6(APC):c.1369del (p.Ser457fs),Deletion,Pathogenic,1,"criteria provided, multiple submitters, no con...",Familial adenomatous polyposis 1|Hereditary ca...,germline;unknown,...,112821950,112821950,na,na,387906229,"Feb 06, 2025",5,3,1,9
1505,15835,796,APC,NM_000038.6(APC):c.1500T>G (p.Tyr500Ter),single nucleotide variant,Pathogenic,1,"criteria provided, multiple submitters, no con...",Familial adenomatous polyposis 1|not provided|...,germline;unknown,...,112827199,112827199,na,na,387906230,"Nov 08, 2024",5,3,1,9
1507,15836,797,APC,NM_000038.6(APC):c.1240C>T (p.Arg414Cys),single nucleotide variant,Benign,1,reviewed by expert panel,Gardner syndrome|not provided|Hereditary cance...,germline;unknown,...,112819272,112819272,na,na,137854567,"Feb 18, 2023",0,4,1,5
1509,15837,798,APC,NM_000038.6(APC):c.904C>T (p.Arg302Ter),single nucleotide variant,Pathogenic/Likely pathogenic,1,"criteria provided, multiple submitters, no con...",Familial adenomatous polyposis 1|Gardner syndr...,germline;somatic;unknown,...,112815564,112815564,na,na,137854568,"Apr 08, 2025",5,3,1,9


In [26]:
def assign_priority_tier(score):
    """
    Convert educational priority score into a simple tier.
    """
    if score >= 9:
        return "Tier 1 - Highest educational priority"
    elif score >= 7:
        return "Tier 2 - High educational priority"
    elif score >= 5:
        return "Tier 3 - Moderate educational priority"
    elif score >= 3:
        return "Tier 4 - Lower educational priority"
    else:
        return "Tier 5 - Minimal educational priority"


priority_df["Educational_Priority_Tier"] = priority_df[
    "Educational_Priority_Score"
].apply(assign_priority_tier)

priority_df = priority_df.sort_values(
    [
        "Educational_Priority_Score",
        "ClinicalSignificance_Score",
        "ReviewStatus_Score",
        "GeneSymbol",
    ],
    ascending=[False, False, False, True],
)

priority_df.head(25)

,#AlleleID,VariationID,GeneSymbol,Name,Type,ClinicalSignificance,ClinSigSimple,ReviewStatus,PhenotypeList,Origin,...,Stop,ReferenceAllele,AlternateAllele,RS# (dbSNP),LastEvaluated,ClinicalSignificance_Score,ReviewStatus_Score,GeneContext_Score,Educational_Priority_Score,Educational_Priority_Tier
12973,22144,7105,CFTR,NM_000492.3(CFTR):c.1521_1523del (p.Phe508del),Deletion,Pathogenic,1,practice guideline,Cystic fibrosis|Bronchiectasis with or without...,germline;inherited;maternal;paternal;unknown,...,117559593,na,na,113993960,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
12975,22145,7106,CFTR,NM_000492.4(CFTR):c.1516ATC[1] (p.Ile507del),Microsatellite,Pathogenic,1,practice guideline,Cystic fibrosis|not provided|not specified|Cys...,germline;unknown,...,117559589,na,na,121908745,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
12981,22148,7109,CFTR,NM_000492.4(CFTR):c.350G>A (p.Arg117His),single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|Congenital bilateral aplasia o...,germline;maternal;paternal;unknown,...,117530975,na,na,78655421,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
12983,22149,7110,CFTR,NM_000492.4(CFTR):c.1040G>C (p.Arg347Pro),single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|Congenital bilateral aplasia o...,germline;paternal;unknown,...,117540270,na,na,77932196,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
12985,22150,7111,CFTR,NM_000492.4(CFTR):c.1364C>A (p.Ala455Glu),single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|ivacaftor response - Efficacy|...,germline;unknown,...,117548795,na,na,74551128,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
12987,22151,7112,CFTR,NM_000492.4(CFTR):c.1585-1G>A,single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|not provided|Cystic fibrosis;C...,germline;unknown,...,117587738,na,na,76713772,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
12989,22152,7113,CFTR,NM_000492.4(CFTR):c.1679G>C (p.Arg560Thr),single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|not provided|not specified|Cys...,germline;unknown,...,117587833,na,na,80055610,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
12993,22154,7115,CFTR,NM_000492.4(CFTR):c.1624G>T (p.Gly542Ter),single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|not provided|Hereditary pancre...,germline;inherited;paternal;unknown,...,117587778,na,na,113993959,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
13003,22159,7120,CFTR,NM_000492.4(CFTR):c.1652G>A (p.Gly551Asp),single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|Hereditary pancreatitis|ivacaf...,germline;maternal;unknown,...,117587806,na,na,75527207,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
13005,22161,7122,CFTR,NM_000492.4(CFTR):c.1657C>T (p.Arg553Ter),single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|not provided|Congenital bilate...,germline;maternal;paternal;unknown,...,117587811,na,na,74597325,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority


In [27]:
priority_tier_counts = (
    priority_df["Educational_Priority_Tier"]
    .value_counts()
    .rename_axis("Educational_Priority_Tier")
    .reset_index(name="Record_Count")
)

priority_tier_counts

,Educational_Priority_Tier,Record_Count
0,Tier 3 - Moderate educational priority,46489
1,Tier 4 - Lower educational priority,45234
2,Tier 1 - Highest educational priority,13816
3,Tier 2 - High educational priority,9327
4,Tier 5 - Minimal educational priority,5189


In [28]:
priority_by_gene = (
    priority_df
    .groupby(["GeneSymbol", "Educational_Priority_Tier"])
    .size()
    .reset_index(name="Record_Count")
    .sort_values(["GeneSymbol", "Record_Count"], ascending=[True, False])
)

priority_by_gene.head(50)

,GeneSymbol,Educational_Priority_Tier,Record_Count
3,APC,Tier 4 - Lower educational priority,7522
2,APC,Tier 3 - Moderate educational priority,5585
4,APC,Tier 5 - Minimal educational priority,1391
1,APC,Tier 2 - High educational priority,1313
0,APC,Tier 1 - Highest educational priority,1058
8,APOB,Tier 4 - Lower educational priority,3010
7,APOB,Tier 3 - Moderate educational priority,1860
6,APOB,Tier 2 - High educational priority,121
9,APOB,Tier 5 - Minimal educational priority,58
5,APOB,Tier 1 - Highest educational priority,45


In [29]:
priority_df.to_csv(
    results_dir / "clinvar_educational_variant_priority_table.csv",
    index=False
)

priority_tier_counts.to_csv(
    results_dir / "clinvar_educational_priority_tier_counts.csv",
    index=False
)

priority_by_gene.to_csv(
    results_dir / "clinvar_educational_priority_by_gene.csv",
    index=False
)

In [30]:
print(f"Rows in priority table: {priority_df.shape[0]:,}")
print(f"Unique genes: {priority_df['GeneSymbol'].nunique():,}")
print(f"Unique VariationIDs: {priority_df['VariationID'].nunique():,}")
print(f"Unique AlleleIDs: {priority_df['#AlleleID'].nunique():,}")

Rows in priority table: 120,055
Unique genes: 18
Unique VariationIDs: 120,055
Unique AlleleIDs: 120,055


In [31]:
priority_df[priority_df["GeneSymbol"] == "CFTR"][
    [
        "#AlleleID",
        "VariationID",
        "GeneSymbol",
        "Name",
        "Type",
        "ClinicalSignificance",
        "ReviewStatus",
        "PhenotypeList",
    ]
].head(20)

,#AlleleID,VariationID,GeneSymbol,Name,Type,ClinicalSignificance,ReviewStatus,PhenotypeList
12973,22144,7105,CFTR,NM_000492.3(CFTR):c.1521_1523del (p.Phe508del),Deletion,Pathogenic,practice guideline,Cystic fibrosis|Bronchiectasis with or without...
12975,22145,7106,CFTR,NM_000492.4(CFTR):c.1516ATC[1] (p.Ile507del),Microsatellite,Pathogenic,practice guideline,Cystic fibrosis|not provided|not specified|Cys...
12981,22148,7109,CFTR,NM_000492.4(CFTR):c.350G>A (p.Arg117His),single nucleotide variant,Pathogenic,practice guideline,Cystic fibrosis|Congenital bilateral aplasia o...
12983,22149,7110,CFTR,NM_000492.4(CFTR):c.1040G>C (p.Arg347Pro),single nucleotide variant,Pathogenic,practice guideline,Cystic fibrosis|Congenital bilateral aplasia o...
12985,22150,7111,CFTR,NM_000492.4(CFTR):c.1364C>A (p.Ala455Glu),single nucleotide variant,Pathogenic,practice guideline,Cystic fibrosis|ivacaftor response - Efficacy|...
12987,22151,7112,CFTR,NM_000492.4(CFTR):c.1585-1G>A,single nucleotide variant,Pathogenic,practice guideline,Cystic fibrosis|not provided|Cystic fibrosis;C...
12989,22152,7113,CFTR,NM_000492.4(CFTR):c.1679G>C (p.Arg560Thr),single nucleotide variant,Pathogenic,practice guideline,Cystic fibrosis|not provided|not specified|Cys...
12993,22154,7115,CFTR,NM_000492.4(CFTR):c.1624G>T (p.Gly542Ter),single nucleotide variant,Pathogenic,practice guideline,Cystic fibrosis|not provided|Hereditary pancre...
13003,22159,7120,CFTR,NM_000492.4(CFTR):c.1652G>A (p.Gly551Asp),single nucleotide variant,Pathogenic,practice guideline,Cystic fibrosis|Hereditary pancreatitis|ivacaf...
13005,22161,7122,CFTR,NM_000492.4(CFTR):c.1657C>T (p.Arg553Ter),single nucleotide variant,Pathogenic,practice guideline,Cystic fibrosis|not provided|Congenital bilate...


In [32]:
priority_unique_variants = (
    priority_df
    .sort_values(
        [
            "Educational_Priority_Score",
            "ReviewStatus_Score",
            "ClinicalSignificance_Score",
        ],
        ascending=[False, False, False]
    )
    .drop_duplicates(subset=["VariationID"])
    .copy()
)

print(f"Original priority rows: {priority_df.shape[0]:,}")
print(f"Unique prioritized variants: {priority_unique_variants.shape[0]:,}")

priority_unique_variants.head(25)

Original priority rows: 120,055
Unique prioritized variants: 120,055


,#AlleleID,VariationID,GeneSymbol,Name,Type,ClinicalSignificance,ClinSigSimple,ReviewStatus,PhenotypeList,Origin,...,Stop,ReferenceAllele,AlternateAllele,RS# (dbSNP),LastEvaluated,ClinicalSignificance_Score,ReviewStatus_Score,GeneContext_Score,Educational_Priority_Score,Educational_Priority_Tier
12973,22144,7105,CFTR,NM_000492.3(CFTR):c.1521_1523del (p.Phe508del),Deletion,Pathogenic,1,practice guideline,Cystic fibrosis|Bronchiectasis with or without...,germline;inherited;maternal;paternal;unknown,...,117559593,na,na,113993960,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
12975,22145,7106,CFTR,NM_000492.4(CFTR):c.1516ATC[1] (p.Ile507del),Microsatellite,Pathogenic,1,practice guideline,Cystic fibrosis|not provided|not specified|Cys...,germline;unknown,...,117559589,na,na,121908745,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
12981,22148,7109,CFTR,NM_000492.4(CFTR):c.350G>A (p.Arg117His),single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|Congenital bilateral aplasia o...,germline;maternal;paternal;unknown,...,117530975,na,na,78655421,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
12983,22149,7110,CFTR,NM_000492.4(CFTR):c.1040G>C (p.Arg347Pro),single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|Congenital bilateral aplasia o...,germline;paternal;unknown,...,117540270,na,na,77932196,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
12985,22150,7111,CFTR,NM_000492.4(CFTR):c.1364C>A (p.Ala455Glu),single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|ivacaftor response - Efficacy|...,germline;unknown,...,117548795,na,na,74551128,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
12987,22151,7112,CFTR,NM_000492.4(CFTR):c.1585-1G>A,single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|not provided|Cystic fibrosis;C...,germline;unknown,...,117587738,na,na,76713772,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
12989,22152,7113,CFTR,NM_000492.4(CFTR):c.1679G>C (p.Arg560Thr),single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|not provided|not specified|Cys...,germline;unknown,...,117587833,na,na,80055610,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
12993,22154,7115,CFTR,NM_000492.4(CFTR):c.1624G>T (p.Gly542Ter),single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|not provided|Hereditary pancre...,germline;inherited;paternal;unknown,...,117587778,na,na,113993959,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
13003,22159,7120,CFTR,NM_000492.4(CFTR):c.1652G>A (p.Gly551Asp),single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|Hereditary pancreatitis|ivacaf...,germline;maternal;unknown,...,117587806,na,na,75527207,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority
13005,22161,7122,CFTR,NM_000492.4(CFTR):c.1657C>T (p.Arg553Ter),single nucleotide variant,Pathogenic,1,practice guideline,Cystic fibrosis|not provided|Congenital bilate...,germline;maternal;paternal;unknown,...,117587811,na,na,74597325,"Mar 03, 2004",5,5,1,11,Tier 1 - Highest educational priority


Because the prioritization table is record-level, repeated gene symbols represent different ClinVar variant records within the same gene rather than duplicate gene-level entries.

In [33]:
priority_unique_variants.to_csv(
    results_dir / "clinvar_educational_unique_variant_priority_table.csv",
    index=False
)

## Interpretation of Educational Prioritization

The educational prioritization workflow organized selected ClinVar records into five tiers using submitted clinical significance, review status, and gene context. Most records fell into moderate or lower educational priority tiers, which is consistent with the composition of the selected ClinVar gene panel. The panel contained a large number of variants classified as uncertain significance, likely benign, benign, or conflicting, in addition to pathogenic and likely pathogenic records.

The highest-priority tiers were enriched for records with pathogenic or likely pathogenic clinical significance and stronger review status. These records may be the most useful examples for educational review because they combine clinically important submitted interpretations with higher-confidence ClinVar review categories.

The gene-level priority summaries show that clinically important genes can contain records across multiple priority tiers. For example, genes such as APC, BRCA1, BRCA2, CFTR, and mismatch repair genes include a mixture of pathogenic, uncertain, conflicting, benign, and lower-review records. This highlights why variant-level interpretation is necessary; the presence of a variant in a clinically relevant gene is not sufficient on its own to determine clinical importance.

These results demonstrate how public ClinVar annotations can be structured into an interpretable prioritization workflow for exploratory review. The priority tiers are not independent clinical classifications and should not be interpreted as diagnostic conclusions. Instead, they provide a reproducible way to organize ClinVar records for educational analysis, portfolio demonstration, and downstream review.

## Record-Level vs Unique Variant-Level Tables

ClinVar variant summary data may contain multiple records associated with the same gene or variant context. The main prioritization table is record-level, while the following table collapses the output to one row per unique ClinVar VariationID for easier variant-level review.

## Project Summary

This notebook analyzed public ClinVar variant summary data across a selected germline clinical gene panel. After filtering to GRCh38 and selected genes, the workflow summarized variant records by gene, clinical significance, review status, pathogenic/likely pathogenic representation, and educational priority tier.

The results show that ClinVar records are highly heterogeneous across clinically relevant genes. Many records fall into uncertain, conflicting, benign, or lower-review categories, reinforcing the importance of variant-level interpretation rather than gene-level assumptions alone.

The educational prioritization table demonstrates how public clinical genomics annotations can be organized into a reproducible review workflow. These tiers are not clinical classifications and should not be used for diagnosis or clinical reporting.